In [2]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
mkdir -p "$work"

export PATH="$work/bin:$PATH"
if ! command -v uv >/dev/null; then
    curl -fsSL https://astral.sh/uv/install.sh | UV_UNMANAGED_INSTALL="$work/bin" sh
fi
export MPLBACKEND=Agg TORCH_HOME="$work/torch-cache"
cd "$work"

repo="$work/ModelAngelo"
if [ ! -d "$repo" ]; then
    git clone https://github.com/3dem/model-angelo.git "$repo"
fi
git -C "$repo" checkout --quiet --detach 994945b

env="$work/ModelAngelo-env"
index=(--index-url https://download.pytorch.org/whl/cu126)
if [ "$(uname -s)" = Darwin ]; then index=(); fi
if [ ! -x "$env/bin/python" ]; then uv venv --python 3.11 "$env"; fi
uv pip install --python "$env/bin/python" "${index[@]}" torch==2.9.1 torchvision==0.24.1
uv pip install --python "$env/bin/python" "$repo"
"$env/bin/model_angelo" setup_weights --bundle-name nucleotides


Using Python 3.11.18 environment at: ModelAngelo-env
Checked 2 packages in 14ms
Using Python 3.11.18 environment at: ModelAngelo-env
Resolved 22 packages in 629ms
Prepared 1 package in 277ms
Uninstalled 1 package in 22ms
Installed 1 package in 45ms
 ~ model-angelo==1.0.18 (from file:///workspace/mica-benchmark/archives/cryo-benchmark/ModelAngelo)


Please make sure you have set the environment variable TORCH_HOME 
to a suitable directory, visible to all relevant users!
Setting up bundle with name: nucleotides for the first time.


100%|██████████| 1.98G/1.98G [05:15<00:00, 6.75MB/s]


Checking checksum of downloaded file.
Checksum of downloaded file matches expected checksum.
Bundle nucleotides successfully installed.
Setting up language model bundle with name: esm1b_t33_650M_UR50S for the first time.


100%|██████████| 7.29G/7.29G [04:05<00:00, 31.9MB/s] 
100%|██████████| 3.10k/3.10k [00:00<00:00, 2.51MB/s]


Successful!
Path to model is /workspace/mica-benchmark/archives/cryo-benchmark/torch-cache/hub/checkpoints/model_angelo_v1.0/nucleotides


In [3]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

method=ModelAngelo
device=0
if [ "$(uname -s)" = Darwin ]; then device=cpu; fi
export MPLBACKEND=Agg TORCH_HOME="$work/torch-cache"

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"

    time "$work/ModelAngelo-env/bin/model_angelo" build -v "${map[0]}" -pf "$fasta" -o "$out" -d "$device"
done


---------------------------- ModelAngelo -----------------------------
By Kiarash Jamali, Scheres Group, MRC Laboratory of Molecular Biology
--------------------- Initial C-alpha prediction ---------------------


/workspace/mica-benchmark/archives/cryo-benchmark/ModelAngelo-env/lib/python3.11/site-packages/model_angelo/c_alpha/inference.py:164: UserWarning: The file /workspace/mica-benchmark/data/targets/22XP/inputs/emd_68757.map does not end with '.mrc'
Please make sure it is an MRC file.
  warnings.warn(f"The file {args.map_path} does not end with '.mrc'\nPlease make sure it is an MRC file.")


100%|██████████| 343/343 [04:03<00:00,  1.41it/s]
------------------ GNN model refinement, round 1 / 3 ------------------


/workspace/mica-benchmark/archives/cryo-benchmark/ModelAngelo-env/lib/python3.11/site-packages/model_angelo/gnn/inference.py:105: UserWarning: The file /workspace/mica-benchmark/data/targets/22XP/inputs/emd_68757.map does not end with '.mrc'
Please make sure it is an MRC file.
  warnings.warn(f"The file {args.map} does not end with '.mrc'\nPlease make sure it is an MRC file.")


100%|██████████| 1019/1019 [01:56<00:00,  8.71it/s]
------------------ GNN model refinement, round 2 / 3 ------------------


/workspace/mica-benchmark/archives/cryo-benchmark/ModelAngelo-env/lib/python3.11/site-packages/model_angelo/gnn/inference.py:105: UserWarning: The file /workspace/mica-benchmark/data/targets/22XP/inputs/emd_68757.map does not end with '.mrc'
Please make sure it is an MRC file.
  warnings.warn(f"The file {args.map} does not end with '.mrc'\nPlease make sure it is an MRC file.")


... (10 outputs left out)

100%|██████████| 734/734 [01:20<00:00,  9.09it/s]
------------------ GNN model refinement, round 3 / 3 ------------------


/workspace/mica-benchmark/archives/cryo-benchmark/ModelAngelo-env/lib/python3.11/site-packages/model_angelo/gnn/inference.py:105: UserWarning: The file /workspace/mica-benchmark/data/targets/32MC/inputs/emd_58994.map does not end with '.mrc'
Please make sure it is an MRC file.
  warnings.warn(f"The file {args.map} does not end with '.mrc'\nPlease make sure it is an MRC file.")


100%|██████████| 730/730 [01:07<00:00, 10.88it/s]
----------------------------------------------------------------------
ModelAngelo build has been completed successfully!
----------------------------------------------------------------------
You can find your output mmCIF file here: /workspace/mica-benchmark/archives/cryo-benchmark/runs/ModelAngelo/32MC/32MC.cif
----------------------------------------------------------------------
The raw output without pruning might be useful to show 
some parts of the map that may be modelled, 
but could not be automatically modelled. 
However, the amino acid classifications are generally 
not going to be correct. 
You can find that here: /workspace/mica-benchmark/archives/cryo-benchmark/runs/ModelAngelo/32MC/32MC_raw.cif
----------------------------------------------------------------------
(Experimental) We now have CIF files with entropy scores 
for each residue. These are useful for identifying regions 
of the map that have lower confidence in 


real	7m0.614s
user	56m6.836s
sys	1m44.471s


In [4]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="ModelAngelo"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$out/$target.cif"
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done


Cloning into '/workspace/mica-benchmark/archives/cryo-benchmark/USalign'...


make: Entering directory '/workspace/mica-benchmark/archives/cryo-benchmark/USalign'
g++ -O3 -ffast-math USalign.cpp -o USalign 
make: Leaving directory '/workspace/mica-benchmark/archives/cryo-benchmark/USalign'
